# SQL query from table names - Continued

In [11]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [12]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [1]:
context = [ {'role':'system', 'content':"""
 CREATE SEVERAL (4+) TABLES HERE

## Employee Directory

| Name         | Department   | Email                |
|--------------|--------------|----------------------|
| Emma Lee     | HR           | emma.lee@company.com |
| Javier Cruz  | IT           | javier.cruz@company.com |
| Priya Singh  | Marketing    | priya.singh@company.com |

## Product Inventory

| Product Name   | SKU     | Stock | Price   |
|----------------|---------|-------|---------|
| LED Monitor    | MON123  | 30    | $120.00 |
| Wireless Mouse | MOU456  | 85    | $25.99  |
| Laptop Stand   | LAP789  | 20    | $32.50  |

## Event Schedule

| Event            | Date         | Location              |
|------------------|--------------|-----------------------|
| Kickoff Meeting  | Nov 10, 2025 | Main Conference Room  |
| Workshop         | Nov 12, 2025 | Online Webinar        |
| Feedback Session | Nov 15, 2025 | Main Conference Room  |

## Customer Feedback

| Customer       | Rating | Feedback                   |
|----------------|--------|----------------------------|
| Sarah Mathews  | 5      | Great experience!          |
| Daniel Abbott  | 4      | Quick service              |
| Maya Ali       | 3      | Room for improvement       |
""" } ]



In [7]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
Give me the customer withthe highest rating
"""
})


Query result: Customer with the highest rating displayed above.


In [9]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [15]:
#new
context_user = context.copy()
print(return_CCRMSQL("""YOUR QUERY HERE""", context_user))

SELECT Customer
FROM Customer_Feedback
ORDER BY Rating DESC
LIMIT 1;


In [17]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("customer with highest rating", old_context_user))

This is your SQL:
```sql
SELECT * 
FROM customers
ORDER BY rating DESC
LIMIT 1;
```

This SQL query selects all columns from the customers table, orders the results by rating in descending order (highest rating first), and limits the output to only the first row, which corresponds to the customer with the highest rating.


In [22]:
#new
print(return_CCRMSQL("customer with highest rating", context_user))

The customer with the highest rating is Sarah Mathews.


In [23]:
#old
print(return_CCRMSQL("give me all the products", old_context_user))

This is your SQL:
```sql
SELECT * FROM products;
```
This SQL command retrieves all the records from the "products" table.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [25]:
#new
print(return_CCRMSQL("Most expensive product", context_user))

SELECT *
FROM Product_Inventory
ORDER BY Price DESC
LIMIT 1;


In [26]:
#new
print(return_CCRMSQL("dates for all events", context_user))

SELECT Event, Date
FROM Event_Schedule;


In [27]:
#new
print(return_CCRMSQL("total number of products", context_user))

```sql
SELECT COUNT(*)
FROM Product_Inventory;
```
